# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Chạy trên **Colab T4** (trình duyệt hoặc extension Colab trong VS Code): chọn kernel Colab GPU T4 rồi **Run All**.
Mã nằm trong `lab10.py`. Mọi bước đều chạy lại được: thí nghiệm nào đã có `report/evidence/<tên>.json` thì bỏ qua.
Sau mỗi thí nghiệm, notebook in ra một gói `report/` (base64). Lưu notebook (Ctrl+S) rồi chạy `python extract_outputs.py` ở máy local để lấy file.

## 0. Môi trường (Colab T4)

In [ ]:
import os, sys, subprocess
REPO = 'https://github.com/long2110d/K4-Track-4-Day10-VisionLanguageModel.git'
USE_DRIVE = False   # True: lưu work/ sang Google Drive (chỉ khi chạy Colab trên trình duyệt)

BASE = '/content/lab10' if os.path.exists('/content') else os.getcwd()
if not os.path.exists(os.path.join(BASE, 'lab10.py')):
    subprocess.run(['git', 'clone', REPO, BASE], check=True)
else:
    subprocess.run(['git', '-C', BASE, 'pull', '--ff-only'], check=False)
os.chdir(BASE)
sys.path.insert(0, BASE)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/lab10_work', exist_ok=True)
    if not os.path.exists('work'):
        os.symlink('/content/drive/MyDrive/lab10_work', 'work')
print('thư mục làm việc:', os.getcwd())

In [ ]:
!nvidia-smi -L
!pip -q install -r requirements-colab.txt kagglehub matplotlib
!python -m pytest -q
!python -m pytest -q test_lab10.py

## 1. Dữ liệu: tải và chia split
Dataset Kaggle công khai nên `kagglehub` tải được không cần khoá. Nếu lỗi, đặt biến môi trường `KAGGLE_USERNAME`/`KAGGLE_KEY` rồi chạy lại.

In [ ]:
if not os.path.exists('data/splits/meta.json'):
    import kagglehub
    src = kagglehub.dataset_download('lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes')
    !python scripts/prepare_data.py --source "{src}"
import json, lab10
from lab10 import *
sessions = split_stats()
print(data_stats())
train_rows, test_rows = load_split('train'), load_split('test')
show_grid([draw_boxes(e, title=e['image_id'][:40]) for e in train_rows[::240][:6]],
          path='report/figures/train_samples.jpg')

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from scripts.score import _gt_boxes
e = next(r for r in train_rows if len(r['objects']) >= 2)
ans = objects_to_answer(e['objects'])
print(ans)
# quy ngược về pixel và đối chiếu với nhãn YOLO gốc (sai số chỉ do làm tròn)
gt = sorted((g['bbox'] for g in _gt_boxes(e)), key=lambda b: (round(b[1]), round(b[0])))
for b, g in zip(json.loads(ans), gt):
    print(b['label'], [round(v, 1) for v in norm1000_to_pixel(b['bbox_2d'], 640, 360)],
          [round(v, 1) for v in norm1000_to_pixel(g, 640, 360)])
print(PROMPT)
draw_boxes(e, raw=ans, title='xanh = GT, do = JSON da quy doi')

## 3. E0: zero-shot

In [ ]:
# nạp model 4-bit (ViT + lm_head fp16), đếm tham số từng khối, kiểm tra mask loss, suy luận test, chấm
run_e0()
print(json.load(open('report/extra/model_blocks.json')))
for r in json.load(open('report/extra/e0_samples.json'))['first']:
    print(repr(r['raw'][:300]))
bundle_outputs()

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
`encode_example` (trong `lab10.py`) đã chạy ở ô E0 (`check_example`): token ảnh phải là **220**, phần `labels != -100` giải mã ra đúng JSON + `<|im_end|>`.

In [ ]:
tok = json.load(open('report/extra/token_stats.json'))
print(tok)
assert tok['image'] == 220, tok

## 5. Setup LoRA / đóng băng cho E1, E2, E3
`apply_setup`: E1 LoRA trên 252 module LLM; E2 + projector fp32 train đầy đủ; E3 + LoRA trên 96 module ViT (tổng 348).

In [ ]:
print('LLM :', LLM_TARGETS)
print('ViT :', VIT_TARGETS)
print('projector:', PROJECTOR_KEYS)

## 6. Vòng huấn luyện (mỗi setup ≈ 20–40 phút + ≈ 10 phút suy luận)

In [ ]:
run_experiment('e1', 'E1')
bundle_outputs()

In [ ]:
run_experiment('e2', 'E2')
bundle_outputs()

In [ ]:
run_experiment('e3', 'E3')
bundle_outputs()

## 7. Suy luận và chấm E1–E3

In [ ]:
for k in ('e0', 'e1', 'e2', 'e3'):
    d = json.load(open(f'report/evidence/{k}.json'))
    print(k.upper(), 'F1=%.3f' % d['micro']['f1'], 'CI95=[%.3f, %.3f]' % tuple(d['f1_ci95']),
          'valid=%.3f' % d['parse_valid_rate'], 'meta=', d['meta'])
BEST = best_setup()
coord_check(BEST.lower())     # Câu 3: chấm sai thang toạ độ

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
E4: đổi **rank LoRA** trên setup tốt nhất của E1–E3, r = 4 và r = 64 (alpha = 2r). E5: không làm.

In [ ]:
for r in (4, 64):
    run_experiment(f'e4_rank_{r}', BEST, r=r, alpha=2 * r)
    bundle_outputs()

In [ ]:
make_figures(BEST.lower())
plot_losses()
!python scripts/check_submission.py
bundle_outputs()   # gói cuối cùng: evidence + figures + extra